# 📉 Principal Component Analysis (PCA): Visualization Lab

Welcome to the PCA Lab. We will visualize dimensionality reduction, variance explained, and image compression.

## 1. Theory Overview

PCA finds orthogonal axes of maximum variance, allowing us to project high-dimensional data into lower dimensions while retaining as much information as possible.

## 2. Visualization Playground
Let's see how PCA rotates a 2D dataset to align with the axes of maximum variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Generate correlated 2D data
rng = np.random.RandomState(1)
X = np.dot(rng.rand(2, 2), rng.randn(2, 200)).T

pca = PCA(n_components=2)
pca.fit(X)

def draw_vector(v0, v1, ax=None):
    ax = ax or plt.gca()
    arrowprops=dict(arrowstyle='->', linewidth=3, shrinkA=0, shrinkB=0, color='red')
    ax.annotate('', v1, v0, arrowprops=arrowprops)

plt.scatter(X[:, 0], X[:, 1], alpha=0.5, s=30)
for length, vector in zip(pca.explained_variance_, pca.components_):
    v = vector * 3 * np.sqrt(length)
    draw_vector(pca.mean_, pca.mean_ + v)
plt.axis('equal')
plt.title("Principal Components as Vectors of Maximum Variance")
plt.show()

## 3. From Scratch Implementation
Using Eigen Decomposition.

In [ ]:
X_centered = X - np.mean(X, axis=0)
cov_matrix = np.cov(X_centered, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvectors = eigenvectors[:, sorted_idx]

X_pca_scratch = np.dot(X_centered, eigenvectors)
print("Top Eigenvector (PC1):", eigenvectors[:, 0])

## 4. NumPy Implementation
Same as above.

## 5. Scikit-Learn Implementation
Let's look at the Scree Plot (Cumulative Variance Explained).

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_cancer = StandardScaler().fit_transform(cancer.data)

pca_cancer = PCA().fit(X_cancer)
exp_var = pca_cancer.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(exp_var)+1), exp_var, alpha=0.5, align='center', label='Individual variance')
plt.step(range(1, len(cum_var)+1), cum_var, where='mid', label='Cumulative variance', color='r')
plt.axhline(y=0.95, color='k', linestyle='--', label='95% Threshold')
plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.title('Scree Plot: Breast Cancer Dataset (30 Features)')
plt.legend(loc='best')
plt.show()

## 6. Hyperparameter Impact Study
Visualizing the data compressed to 2D.

In [ ]:
X_cancer_pca = PCA(n_components=2).fit_transform(X_cancer)

plt.figure(figsize=(8, 6))
plt.scatter(X_cancer_pca[:, 0], X_cancer_pca[:, 1], c=cancer.target, cmap='Set1', alpha=0.7)
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Breast Cancer Dataset projected to 2D via PCA')
plt.show()

## 7. Real Dataset Case Study (Image Compression)
Using Labeled Faces in the Wild (LFW).

In [ ]:
from sklearn.datasets import fetch_lfw_people

# Fetch faces (this might take a moment to download initially)
faces = fetch_lfw_people(min_faces_per_person=60)

# PCA for compression
pca_faces = PCA(150, svd_solver='randomized', random_state=42)
pca_faces.fit(faces.data)
components = pca_faces.transform(faces.data)
projected = pca_faces.inverse_transform(components)

# Plot original vs compressed
fig, ax = plt.subplots(2, 5, figsize=(15, 6), subplot_kw={'xticks':[], 'yticks':[]})
for i in range(5):
    ax[0, i].imshow(faces.images[i], cmap='bone')
    ax[0, i].set_title("Original")
    ax[1, i].imshow(projected[i].reshape(faces.images[0].shape), cmap='bone')
    ax[1, i].set_title("150-D Reconstructed")

plt.suptitle("Image Compression via PCA (Original 2914 pixels -> 150 components)")
plt.show()

## 8. Visualization Challenge
Plot the "Eigenfaces" (the actual principal components reshaped back into images).

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(15, 6), subplot_kw={'xticks':[], 'yticks':[]})
for i, ax in enumerate(axes.flat):
    ax.imshow(pca_faces.components_[i].reshape(faces.images[0].shape), cmap='bone')
    ax.set_title(f"Eigenface {i+1}")
plt.suptitle("The Top 24 Eigenfaces")
plt.show()

## 9. Exercises
1. Train a Random Forest on the original LFW dataset vs the PCA compressed dataset. Compare the training time and accuracy.